# Benchmark Evaluation Suite: Automated Model Fine-Tuning, Multi-GPU Execution & MLflow Tracking

This tutorial shows you how to use `run_benchmark.py` to train, evaluate, and compare language models (like ModernGBERT and RoBERTa) across different datasets and hyperparameter settings.

Instead of running scripts manually one by one, you define your models, datasets, learning rates, batch sizes, and random seeds in a single `config.yaml` file. The script reads this file, generates every setting, runs the training jobs in parallel across your available GPUs, and logs all results to a local MLflow database (`mlflow.db`).

---

## Learning Objectives

### This tutorial answers the following questions:
* **Why use a benchmark evaluation suite?**  
  Testing a model once with a single random seed can give misleading results. This suite runs every configuration across multiple random seeds and hyperparameters so you get an accurate, fair comparison of how well models perform.
* **How does hyperparameter grid search work?**  
  You list the learning rates, batch sizes, and seed counts in your YAML config. The code expands these lists into individual tasks, checks if any task was already completed in MLflow, and only runs the remaining ones so you never waste GPU time.
* **How do we evaluate runs and export paper-ready results?**  
  Instead of relying only on default run summaries, the post-processing pipeline (`latex_utils.py`) uses saved parquet artifacts from MLflow (`sqlite:///mlflow.db`), re-computes target metrics (e.g., accuracy, macro F1, ROC-AUC) directly from raw epoch logits, picks the best hyperparameter setting based on validation set performance, and auto-generates LaTeX tables complete with mean and standard deviation ($\text{mean} \pm \text{std}$) across seeds.

### By the end of this tutorial, you will be able to:
* **Run Structured Benchmark Sweeps:** Define models, datasets, and hyperparameters in a single YAML file to execute comprehensive evaluation grids.
* **Analyze Granular Prediction Artifacts:** Inspect item-level model confidence (`prob_X`), logit distributions, and error flags (`is_correct`) using MLflow tables.
* **Generate publication-ready LaTeX tables** by querying local MLflow artifacts, selecting optimal hyperparameter settings via validation performance, and outputting formatted tables with statistical variance across random seeds.
---

## Target Audience

This tutorial is aimed at an **intermediate to advanced level**. You should be familiar with PyTorch, Hugging Face `Transformers`, basic multiprocessing concepts, and Python programming.

## Environment Setup

In [1]:
import os
import yaml
import json
import pandas as pd
from datasets import load_dataset

## Introduction

What exactly is **model benchmarking**?

In model benchmarking, we systematically evaluate machine learning models across various conditions to find out which one performs best on a range of tasks.

For example, suppose we want to compare how well different language models detect hate speech in text. We could easily train a single model on a dataset with a fixed set of settings (e.g., learning rate = `2e-5`, batch size = `16`). Every time the training finishes, we look at the final accuracy score and assume we know which model is better, right? However, this might not be accurate. A model might get lucky on a specific random seed, or perform poorly simply because its learning rate wasn't appropriate. So, evaluating across hyperparameter grids and multiple random seeds provides insights into the statistical distribution.
In this tutorial, we aim for a comprehensive evaluation. Our approach that saves the raw model outputs (raw logits) for every example enabling the researcher to observe model performance across different metric settings.

Benchmarking is transferable to more than just hate speech detection. Researchers and engineers use it to evaluate models for sentiment analysis, stance detection, topic classification, emotion recognition, bias prediction, and many other natural language processing tasks.

## Data Preparation

For this benchmark suite to automatically discover and process your data, all datasets must be stored inside a central `benchmark_datasets/` root directory.

Inside `benchmark_datasets/`, each task or dataset must have its own dedicated subfolder containing three CSV files:
* `train.csv` (used to train the model weights)
* `eval.csv` (used for hyperparameter selection and validation during training)
* `test.csv` (used for final performance evaluation)

### Folder Structure Example

```text
./benchmark_datasets/
├── one_custom_dataset/
│   ├── train.csv
│   ├── val.csv
│   └── test.csv
└── another_custom_dataset/
    ├── train.csv
    ├── val.csv
    └── test.csv
```
### Required CSV Columns

Every CSV file (`train.csv`, `val.csv`, and `test.csv`) must contain at least a text column and a label column. Depending on your task, use the following structure:

* **Single-Text Classification:** Must include `text` and `label` columns.
* **Text-Pair Classification:** Must include `text`, `text2`, and `label` columns.

#### Example Single-Text Format:

| text | label |
| :--- | :--- |
| Yesterday I ate an apple. | about_fruit |
| Yesterday I crashed my Apple. | not_about_fruit |

In [6]:
# 1. Define the output folder path inside benchmark_datasets/
dataset_name = "poem_sentiment"
target_dir = os.path.join("benchmark_datasets", dataset_name)
os.makedirs(target_dir, exist_ok=True)

# 2. Load a small example dataset from Hugging Face (e.g., rotten_tomatoes)
# rotten_tomatoes has 'train', 'validation', and 'test' splits
raw_dataset = load_dataset("mteb/poem_sentiment")

# 3. Convert splits to pandas DataFrames and ensure required column names ('text' and 'label')
df_train = pd.DataFrame(raw_dataset["train"])[["text", "label"]]
df_val = pd.DataFrame(raw_dataset["validation"])[["text", "label"]]
df_test = pd.DataFrame(raw_dataset["test"])[["text", "label"]]

# 4. Save splits to train.csv, eval.csv, and test.csv without index
df_train.to_csv(os.path.join(target_dir, "train.csv"), index=False)
df_val.to_csv(os.path.join(target_dir, "val.csv"), index=False)
df_test.to_csv(os.path.join(target_dir, "test.csv"), index=False)

print(f"Successfully saved '{dataset_name}' dataset to: {target_dir}")
print(f"- train.csv: {len(df_train)} rows, {df_train.columns} columns")
print(f"- val.csv:  {len(df_val)} rows, {df_val.columns} columns")
print(f"- test.csv:  {len(df_test)} rows, {df_test.columns} columns")

Successfully saved 'poem_sentiment' dataset to: benchmark_datasets/poem_sentiment
- train.csv: 885 rows, Index(['text', 'label'], dtype='object') columns
- val.csv:  104 rows, Index(['text', 'label'], dtype='object') columns
- test.csv:  104 rows, Index(['text', 'label'], dtype='object') columns


## Selecting Pre-Trained Language Models & Configuring the Benchmark

Rather than running models individually, this benchmark suite uses `config.yaml` to define models, datasets, and execution settings in one central file.

The orchestrator script (`run_benchmark.py`) parses this YAML file, automatically expands every combination into an execution grid and manages the run in parallel.

---

### What is a Pre-Trained Language Model?

A pre-trained language model (such as **ModernBERT**) has been trained on billions of words using self-supervised tasks (like predicting masked words in sentences). Through this process, the model learns grammar, syntax, context, and semantic relationships across one or many languages.

Because it already understands language structure, we can fine-tune it on downstream classification tasks (e.g., hate speech detection) using relatively small datasets.

---

### Understanding Model Selection in `config_evaluation.yaml`

In the configuration file, we specify our execution settings, target models, hyperparameters, and datasets. Each model entry maps a human-readable `display_name` to its Hugging Face checkpoint (`model_link` and `model_type`):

```yaml
settings:
  CUDA_VISIBLE_DEVICES: "0"
  num_workers: 4
  output_dir: "./evaluations"
  data_dir: "./benchmark_datasets"
  early_stopping: false

models:
  - display_name: "BERT"
    model_link: "bert-base-cased"
    model_type: "bert-base-cased"
  - display_name: "ModernBERT"
    model_link: "answerdotai/ModernBERT-base"
    model_type: "answerdotai/ModernBERT-base"

hyperparameters:
  lr: ["3e-5"]
  bsz: [16, 32]
  num_seeds: 5
  epochs: [3]

datasets:
  - name: "Poem Sentiment"
    description: "Classify the sentiment of poem verses."
    directory: "/poem_sentiment/"
    tokenization: "one_text"
    max_length: 128
```

### Execution & Grid Search Expansion

When `run_benchmark.py` parses your configuration file, it computes the Cartesian product across all defined models, datasets, hyperparameters, and random seeds.

The grid expansion yields **40 individual fine-tuning runs**:

$$\text{2 Models} \times \text{1 Dataset} \times \text{1 Learning Rate} \times \text{2 Batch Sizes} \times \text{1 Epoch Setting} \times \text{5 Seeds} = 20 \text{ Tasks}$$

#### Grid Breakdown

* **RoBERTa (20 Tasks):**
  * `bsz = 16`, `lr = 3e-5`, `epochs = 3` $\rightarrow$ Seeds 0, 1, 2, 3, 4 (5 runs)
  * `bsz = 32`, `lr = 3e-5`, `epochs = 3` $\rightarrow$ Seeds 0, 1, 2, 3, 4 (5 runs)
* **BERT (20 Tasks):**
  * `bsz = 16`, `lr = 3e-5`, `epochs = 3` $\rightarrow$ Seeds 0, 1, 2, 3, 4 (5 runs)
  * `bsz = 32`, `lr = 3e-5`, `epochs = 3` $\rightarrow$ Seeds 0, 1, 2, 3, 4 (5 runs)

#### Execution Behavior

* **GPU Allocation:** Runs sequentially on GPU `0` as specified by `CUDA_VISIBLE_DEVICES: "0"`.
* **Deduplication & Resume:** Before executing each run, `run_benchmark.py` queries `sqlite:///mlflow.db`. If a run with identical hyperparameters and seed is already logged as `FINISHED`, it is skipped automatically.

## Running the Benchmark & Tracking Results

Once your data is prepared and your config_evaluation.yaml file is configured, you can launch the automated training queue with a single command:



In [ ]:
!python run_benchmark.py --config tutorial_config.yaml # Exclude models that you dont want to test

The orchestrator script will initialize, parse your YAML configuration, create the execution grid, and begin sequentially training and evaluating each model combination while logging metrics directly to MLflow.

---

## Visualizing Results with the MLflow UI

All hyperparameter runs, evaluation metrics (e.g., accuracy, macro F1, loss), and model artifacts are automatically tracked inside your local SQLite database (sqlite:///mlflow.db).

### 1. Launching the Interface

To launch the MLflow web interface, open a terminal window in your project root directory and run:

By default, MLflow starts a local web server at [http://127.0.0.1:5000](https://www.google.com/search?q=http://127.0.0.1:5000&utm_source=gemini). Open this URL in your browser to view the interface.

In [ ]:
!mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000


### 2. MLflow Interface Overview

* **Experiments Overview:** Lists all active experiments. Each dataset creates an experiment entry where runs are aggregated.

![image.png](attachment:image.png)

* **Configuration Overview:** Lists all configurations tested for this dataset.

# ![image.png](attachment:image.png)

* **Metric Comparisons:** Lists all metrics logged for the different configurations.

![image.png](attachment:image.png)

* **Comparison Table:** Offers direct comparison per data point for different group settings and logged values.

![image.png](attachment:image.png)

## Results

### Macro Average F1-Score

The classification results are evaluated using the **F1-score**. The F1-score combines **precision** and **recall** into a single metric.

The *precision* tells us “When we predict a label, how often is the prediction correct?”

The *recall* tells us “How many instances of a class do we correctly identify?”

The *F1-score* is the harmonic mean of *precision* and *recall*, balancing both measures.

The *macro F1-score* calculates the F1-score independently for each class and then averages them. It therefore tells us “How well do we classify when each class is given equal importance?”

The results in Table X report the **mean macro F1-score and standard deviation across five runs**. For the Poem Sentiment dataset, BERT achieves $0.8490 \pm 0.0234$, while ModernBERT achieves $0.6326 \pm 0.1008$.


In [ ]:
!python latex_utils.py   --metric_name "f1_macro"   --group_by "model_name"   --tracking_uri "sqlite:///mlflow.db"   --output_file "combined_results.tex"

### Accuracy

The classification results are additionally evaluated using **accuracy**. Accuracy measures the proportion of all predictions that are classified correctly.

The *accuracy* tells us “How many of our predictions are correct?”

The results in Table X report the **mean accuracy and standard deviation across five runs**. For the Poem Sentiment dataset, BERT achieves $0.8827 \pm 0.0219$, while ModernBERT achieves $0.8385 \pm 0.0172$.

In [ ]:
!python latex_utils.py   --metric_name "accuracy"   --group_by "model_name"   --tracking_uri "sqlite:///mlflow.db"   --output_file "combined_results.tex"

### Discussion

Both metrics show higher results for BERT than for ModernBERT. The difference is particularly pronounced for **macro F1**, where BERT achieves $0.8490$ compared to $0.6326$ for ModernBERT. The accuracy results show a smaller difference, with BERT achieving $0.8827$ compared to $0.8385$.

This difference between accuracy and macro F1 suggests that the models' performance varies across the individual sentiment classes, which is reflected more strongly by the macro F1-score than by overall accuracy.
